In [ ]:
# Ô 1 — gỡ torchao, kiểm GPU
import os, subprocess, sys
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # tqdm ngoài terminal làm ngập log, từng treo commit 7 h
os.environ["TQDM_DISABLE"] = "1"
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
r = subprocess.run([sys.executable, "-c",
    "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__, "
    "'| GPU', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"],
    capture_output=True, text=True)
print(r.stdout, r.stderr[-1500:], flush=True)
import torch
NGPU = torch.cuda.device_count()
assert NGPU >= 1, "DỪNG: không thấy GPU — Settings → Accelerator → GPU T4 x2"
print("NGPU =", NGPU, flush=True)


In [ ]:
# Ô 2 — gói mã, 400 bản ghi, 9 tệp câu, tải trước model + cây trợ năng
import os, glob, json, shutil, hashlib, sys
W = "/kaggle/working"

# gói c1-exec8: nhận bằng cặp c1_picks.json + c1_mau.jsonl đứng cạnh nhau
cand = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/c1_picks.json", recursive=True)
               if os.path.exists(os.path.join(os.path.dirname(p), "c1_mau.jsonl"))})
assert len(cand) == 1, f"DỪNG: cần đúng một gói c1-exec8, thấy {cand}"
PK = cand[0]
md5 = hashlib.md5(open(f"{PK}/c1_mau.jsonl", "rb").read()).hexdigest()
assert md5 == "d757554326977309c3a65ae0b144c211", f"DỪNG: c1_mau.jsonl lệch bản máy nhà ({md5})"
WS = f"{W}/pk/thesis"
shutil.copytree(f"{PK}/thesis", WS, dirs_exist_ok=True)
sr = open(f"{WS}/harness/score_run.py", encoding="utf-8").read()
assert "--recs-file" in sr and "TỆP THÔ KHÔNG KHỚP" in sr, "DỪNG: score_run.py là bản cũ"
PICKS = json.load(open(f"{PK}/c1_picks.json", encoding="utf-8"))["picks"]
assert len(PICKS) == 400 and set(PICKS) <= set("012345678"), len(PICKS)

# dữ liệu val: mọi tệp lấy theo thư mục của val_cham600.jsonl
# (gói mã cũng có vài ảnh ep*_s*.png; glob toàn /kaggle/input dễ trỏ nhầm)
v600 = glob.glob("/kaggle/input/**/val_cham600.jsonl", recursive=True)
assert len(v600) == 1, f"DỪNG: cần đúng một dataset thesis-val-cham, thấy {v600}"
BASE = os.path.dirname(v600[0])
assert os.path.exists(f"{BASE}/val_cham400.jsonl")

C1 = [json.loads(l) for l in open(f"{PK}/c1_mau.jsonl", encoding="utf-8")]
assert len(C1) == 400
assert [(r["episode_id"], r["step_id"]) for r in C1[:3]] == [(11948, 3), (2782, 0), (188, 8)]
TAPT = ("click", "long_press")
assert all(len(r["mau"]) == 8 for r in C1)
# 5 mẫu rỗng có thật nhưng đều ở bước không click (không vào phép chấm) ⇒ chỉ kiểm bước click
assert all(r["greedy"] and all(r["mau"]) for r in C1 if r["action_type"] in TAPT), "DỪNG: câu rỗng ở bước click"
KEYS = {(r["episode_id"], r["step_id"]) for r in C1}

VD = f"{W}/c1data"
os.makedirs(VD, exist_ok=True)
recs = {}
for f in ("val_cham400.jsonl", "val_cham600.jsonl"):
    for d in map(json.loads, open(f"{BASE}/{f}", encoding="utf-8")):
        k = (d["episode_id"], d["step_id"])
        if k in KEYS and k not in recs:
            d.setdefault("gold_instruction", d["target_instruction"])
            recs[k] = d
assert len(recs) == 400, f"DỪNG: chỉ ghép được {len(recs)}/400 bước C1"
with open(f"{VD}/c1_recs.jsonl", "w", encoding="utf-8") as fo:
    for r in C1:
        fo.write(json.dumps(recs[(r["episode_id"], r["step_id"])], ensure_ascii=False) + "\n")
shutil.copy(f"{BASE}/ocr.jsonl", f"{VD}/ocr.jsonl")
if os.path.lexists(f"{VD}/images"):
    os.remove(f"{VD}/images")
os.symlink(f"{BASE}/images", f"{VD}/images")

taps = [d for d in recs.values() if d["action"].get("action_type") in TAPT and "x" in d["action"]]
thieu = [d["image"] for d in taps if not os.path.exists(f"{VD}/{d['image']}")]
print("bước C1:", len(recs), "| click:", len(taps), "(cần 249) | ảnh thiếu:", len(thieu), "(cần 0)", flush=True)
assert len(taps) == 249 and not thieu
assert {(d["episode_id"], d["step_id"]) for d in taps} == \
       {(r["episode_id"], r["step_id"]) for r in C1 if r["action_type"] in TAPT}, "DỪNG: action_type lệch"

PD = f"{W}/c1preds"
os.makedirs(PD, exist_ok=True)
for k in range(9):
    with open(f"{PD}/k{k}.jsonl", "w", encoding="utf-8") as fo:
        for r in C1:
            s = r["greedy"] if k == 0 else r["mau"][k - 1]
            fo.write(json.dumps({"episode_id": r["episode_id"], "step_id": r["step_id"], "pred": s},
                                ensure_ascii=False) + "\n")
OUT = f"{W}/c1score"
os.makedirs(OUT, exist_ok=True)
print("9 tệp câu:", sorted(os.listdir(PD)), flush=True)

# tải MỘT lần ở đây, để hai tiến trình GPU không cùng tải một lúc
from huggingface_hub import snapshot_download
print("UGround:", snapshot_download("osunlp/UGround-V1-2B"), flush=True)
sys.path.insert(0, f"{WS}/harness")
import a11y_inventory as A11Y, score_run as SR
names = set(A11Y._zip().namelist())
co_cay = sum(A11Y.key_for(f"episode_{d['episode_id']}_screenshot_{d['step_id']}.png") in names for d in taps)
rong = sum(len(SR.buttons_of(d)) == 0 for d in taps)
print(f"cây trợ năng: {co_cay}/249 bước click · {rong} bước rỗng nút", flush=True)
assert co_cay == 249 and rong == 0, "DỪNG: thiếu cây trợ năng ⇒ Voronoi chấm sai mà không báo"


In [ ]:
# Ô 3 — nối tiếp (nếu có) · thử 3 bước · chấm đủ 9 tệp trên 2 GPU
import subprocess, time, glob, shutil
env = {**os.environ, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1"}

def lenh(k, out):
    return (f"python harness/score_run.py --mode score --grounder uground "
            f"--preds {PD}/k{k}.jsonl --data-root {VD} --recs-file c1_recs.jsonl --out {out}")

# ① nối tiếp: nếu có gắn Output của một version trước làm Input thì chép tệp thô cũ sang
#    (score_run.py tự bỏ bước đã chấm, và tự dừng nếu tệp thô không khớp tệp câu)
for p in glob.glob("/kaggle/input/**/c1score/score_k*_raw.jsonl", recursive=True):
    dst = f"{OUT}/{os.path.basename(p)}"
    if not os.path.exists(dst):
        shutil.copy(p, dst)
        print("nối tiếp từ", p, sum(1 for _ in open(dst)), "bước", flush=True)

# ② thử 3 bước trên GPU 0, ra thư mục riêng — hỏng thì commit dừng ở đây sau vài phút
os.makedirs(f"{W}/thu", exist_ok=True)
with open(f"{W}/thu.log", "w") as lg:
    rc = subprocess.run(lenh(0, f"{W}/thu/score_thu.json") + " --n 3", shell=True, cwd=WS,
                        env={**env, "CUDA_VISIBLE_DEVICES": "0"}, stdout=lg, stderr=subprocess.STDOUT).returncode
tl = open(f"{W}/thu.log").read()
print(tl[-1500:], flush=True)
thu = [json.loads(l) for l in open(f"{W}/thu/score_thu_raw.jsonl")]
assert rc == 0 and len(thu) == 3, f"DỪNG: lượt thử lỗi (mã {rc}, {len(thu)} bước) — xem thu.log"
assert f"[dữ liệu] {VD}/c1_recs.jsonl" in tl and "[uground] nạp osunlp/UGround-V1-2B" in tl, \
    "DỪNG: tiến trình không đọc đúng dữ liệu hoặc đúng bộ trỏ"
assert all(o.get("pred_xy") and o.get("n_buttons", 0) > 0 for o in thu), "DỪNG: bộ trỏ không trả toạ độ hoặc 0 nút"
print("THỬ 3 BƯỚC: ĐẠT", flush=True)

# ③ lượt đủ: k chia đều cho các GPU, mỗi GPU chạy nối đuôi các tệp của mình
chuoi = {g: [k for k in range(9) if k % NGPU == g] for g in range(NGPU)}
PS = [subprocess.Popen(["bash", "-c", " && ".join(lenh(k, f"{OUT}/score_k{k}.json") for k in ks)],
                       cwd=WS, env={**env, "CUDA_VISIBLE_DEVICES": str(g)}, start_new_session=True,
                       stdout=open(f"{W}/cham_gpu{g}.log", "a"), stderr=subprocess.STDOUT)
      for g, ks in chuoi.items()]

def dem():
    return [sum(1 for _ in open(f"{OUT}/score_k{k}_raw.jsonl")) if os.path.exists(f"{OUT}/score_k{k}_raw.jsonl")
            else 0 for k in range(9)]

t0 = time.time()
while any(p.poll() is None for p in PS):
    time.sleep(120)
    print(f"{time.strftime('%H:%M')} - {(time.time()-t0)/3600:4.2f} h - "
          + " ".join(f"k{k}:{n}/249" for k, n in enumerate(dem())), flush=True)
    if time.time() - t0 > 10 * 3600:            # đệm dưới trần 12 h; tệp thô ghi dần nên giữ được phần đã xong
        for p in PS:
            p.terminate()
        print("⛔ quá 10 h — dừng, giữ phần đã ghi", flush=True)

for g in range(NGPU):
    print(f"--- GPU {g} ---\n" + subprocess.run(["tail", "-8", f"{W}/cham_gpu{g}.log"],
                                               capture_output=True, text=True).stdout, flush=True)
rcs = [p.returncode for p in PS]
print("mã thoát:", rcs, "| đếm:", dem(), flush=True)
assert rcs == [0] * NGPU and dem() == [249] * 9, "DỪNG: chưa đủ 249/249 ở mọi tệp — xem §5 (nối tiếp)"


In [ ]:
# Ô 4 — gộp số
import random
R = {}
for k in range(9):
    R[k] = {}
    for line in open(f"{OUT}/score_k{k}_raw.jsonl", encoding="utf-8"):
        o = json.loads(line)
        R[k][(o["episode_id"], o["step_id"])] = o
tap_keys = [(r["episode_id"], r["step_id"]) for r in C1 if (r["episode_id"], r["step_id"]) in R[0]]
assert len(tap_keys) == 249, len(tap_keys)
for k in range(9):
    assert set(R[k]) == set(tap_keys), f"k{k} thiếu bước"

def ex(k, key):
    return int(R[k][key].get("executable", 0))

def disk(k, key):
    return int(R[k][key].get("hit_disk", 0))

idx = {(r["episode_id"], r["step_id"]): i for i, r in enumerate(C1)}
pick = {key: int(PICKS[idx[key]]) for key in tap_keys}
n = len(tap_keys)

def pct(xs):
    return round(100 * sum(xs) / n, 2)

tong = {
    "n_click": n,
    "exec_greedy": pct([ex(0, q) for q in tap_keys]),
    "exec_oracle_spice": pct([ex(pick[q], q) for q in tap_keys]),
    "hit_disk_greedy": pct([disk(0, q) for q in tap_keys]),
    "hit_disk_oracle_spice": pct([disk(pick[q], q) for q in tap_keys]),
    "exec_moi_mau": {f"k{k}": pct([ex(k, q) for q in tap_keys]) for k in range(1, 9)},
    "exec_tb_8_mau": round(sum(pct([ex(k, q) for q in tap_keys]) for k in range(1, 9)) / 8, 2),
    "ti_le_co_it_nhat_1_cau_trung_trong_9": pct([max(ex(k, q) for k in range(9)) for q in tap_keys]),
    "so_buoc_oracle_doi_cau": sum(1 for q in tap_keys if pick[q] != 0),
}
doi = [q for q in tap_keys if pick[q] != 0]
tong["tren_buoc_oracle_doi_cau"] = {
    "n": len(doi),
    "exec_greedy": round(100 * sum(ex(0, q) for q in doi) / max(len(doi), 1), 2),
    "exec_oracle": round(100 * sum(ex(pick[q], q) for q in doi) / max(len(doi), 1), 2),
    "greedy_truot_oracle_trung": sum(1 for q in doi if not ex(0, q) and ex(pick[q], q)),
    "greedy_trung_oracle_truot": sum(1 for q in doi if ex(0, q) and not ex(pick[q], q)),
}
assert tong["so_buoc_oracle_doi_cau"] == 119, "DỪNG: PICKS lệch bản CPU (phải đổi câu ở 119/249 bước click)"

# KTC 95% bootstrap theo episode cho hiệu oracle − greedy
eps = sorted({q[0] for q in tap_keys})
theo_ep = {e: [q for q in tap_keys if q[0] == e] for e in eps}
rng = random.Random(101)
ds = []
for _ in range(10000):
    mau_ep = [rng.choice(eps) for _ in eps]
    qs = [q for e in mau_ep for q in theo_ep[e]]
    ds.append(100 * sum(ex(pick[q], q) - ex(0, q) for q in qs) / len(qs))
ds.sort()
tong["hieu_exec_oracle_tru_greedy"] = round(tong["exec_oracle_spice"] - tong["exec_greedy"], 2)
tong["ktc95_bootstrap_episode"] = [round(ds[249], 2), round(ds[9749], 2)]

print(json.dumps(tong, ensure_ascii=False, indent=1), flush=True)
json.dump(tong, open(f"{W}/c1_exec_tong.json", "w"), ensure_ascii=False, indent=1)


In [ ]:
# Ô 5 — gói kết quả
import zipfile
with zipfile.ZipFile(f"{W}/c1_exec_8mau.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(glob.glob(f"{OUT}/*") + glob.glob(f"{W}/cham_gpu*.log")
                    + [f"{W}/c1_exec_tong.json", f"{W}/thu.log"]):
        if os.path.exists(p):
            z.write(p, os.path.relpath(p, W))
print(os.path.getsize(f"{W}/c1_exec_8mau.zip") // 1024, "KB — tải ở tab Output của version", flush=True)
shutil.rmtree(f"{W}/pk", ignore_errors=True)   # gói mã chép tạm, không cần lên Output
